In [25]:
import os
import shutil
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# ==========================================
# 1. 설정 및 준비
# ==========================================
# 🚨 분류할 이미지가 잔뜩 섞여 있는 원본 폴더 경로
TARGET_FOLDER = r"C:\Users\user\Desktop\musinsa_images\musinsa_images\후드집업"

# 🚨 몇 개의 그룹(폴더)으로 나눌지 설정 (노이즈가 많을수록 숫자를 키우면 좋습니다)
NUM_CLUSTERS = 25

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 {device} 환경에서 폴더 자동 분류를 시작합니다.")

# ==========================================
# 2. 특징 추출 (ResNet50)
# ==========================================
print("\n[1/3] 🧠 AI가 옷들의 생김새를 분석하는 중입니다...")
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Identity() 
model = model.to(device)
model.eval()

image_paths = []
features = []

valid_extensions = (".jpg", ".jpeg", ".png")
all_files = [f for f in os.listdir(TARGET_FOLDER) if f.lower().endswith(valid_extensions)]

for img_name in all_files:
    img_path = os.path.join(TARGET_FOLDER, img_name)
    try:
        img = Image.open(img_path).convert('RGB')
        img_tensor = preprocess(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            feature = model(img_tensor).cpu().numpy().flatten()
            
        features.append(feature)
        image_paths.append(img_path)
    except Exception as e:
        continue

features = np.array(features)
print(f"총 {len(features)}장의 이미지 분석 완료!")

# ==========================================
# 3. K-Means 군집화 (비슷한 것끼리 묶기)
# ==========================================
print(f"\n[2/3] 📊 특징들을 {NUM_CLUSTERS}개의 그룹으로 분류합니다...")

# 연산 속도와 정확도를 높이기 위해 PCA로 중요 정보만 압축
pca = PCA(n_components=50, random_state=42)
reduced_features = pca.fit_transform(features)

# K-Means 알고리즘으로 강제 그룹화
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init='auto')
labels = kmeans.fit_predict(reduced_features)

# ==========================================
# 4. 결과에 따라 폴더로 이동
# ==========================================
print(f"\n[3/3] 📁 분류된 결과에 따라 새 폴더로 사진들을 이사시킵니다...")

# 그룹별 폴더 미리 만들기
for i in range(NUM_CLUSTERS):
    cluster_folder = os.path.join(TARGET_FOLDER, f"Group_{i}")
    os.makedirs(cluster_folder, exist_ok=True)

# 사진들을 각각 배정된 그룹 폴더로 이동 (복사가 아닌 이동)
for idx, label in enumerate(labels):
    src_path = image_paths[idx]
    file_name = os.path.basename(src_path)
    dst_path = os.path.join(TARGET_FOLDER, f"Group_{label}", file_name)
    
    shutil.move(src_path, dst_path)

print(f"\n🎉 분류 완료! {TARGET_FOLDER} 안에 Group_0 부터 Group_{NUM_CLUSTERS-1} 까지 폴더가 생성되었습니다.")
print("이제 폴더를 하나씩 열어보면서, 이상한 사진들만 모인 폴더를 통째로 지워주세요!")

🚀 cuda 환경에서 폴더 자동 분류를 시작합니다.

[1/3] 🧠 AI가 옷들의 생김새를 분석하는 중입니다...
총 3006장의 이미지 분석 완료!

[2/3] 📊 특징들을 25개의 그룹으로 분류합니다...

[3/3] 📁 분류된 결과에 따라 새 폴더로 사진들을 이사시킵니다...

🎉 분류 완료! C:\Users\user\Desktop\musinsa_images\musinsa_images\후드집업 안에 Group_0 부터 Group_24 까지 폴더가 생성되었습니다.
이제 폴더를 하나씩 열어보면서, 이상한 사진들만 모인 폴더를 통째로 지워주세요!


In [2]:
import os
import shutil

# ==========================================
# 1. 설정 (복구할 최상위 폴더 경로)
# ==========================================
# 🚨 원래대로 되돌릴 폴더 경로를 입력하세요.
TARGET_FOLDER = r"C:\Users\user\Desktop\musinsa_images\musinsa_images\가디건"

print(f"🚀 [{TARGET_FOLDER}] 복구(Rollback) 작업을 시작합니다...\n")

# ==========================================
# 2. 파일 꺼내기 및 폴더 삭제
# ==========================================
restored_count = 0
removed_folders = 0

# TARGET_FOLDER 안의 모든 항목(파일/폴더)을 확인
for item in os.listdir(TARGET_FOLDER):
    folder_path = os.path.join(TARGET_FOLDER, item)
    
    # 💡 안전장치: 오직 'Group_'으로 시작하는 폴더만 건드립니다. (다른 폴더 보호)
    if os.path.isdir(folder_path) and item.startswith("Group_"):
        print(f"📂 [{item}] 폴더 해체 중...")
        
        # 1. 폴더 안의 파일들을 원래 위치(TARGET_FOLDER)로 모두 이동
        for file_name in os.listdir(folder_path):
            src_path = os.path.join(folder_path, file_name)
            dst_path = os.path.join(TARGET_FOLDER, file_name)
            
            try:
                shutil.move(src_path, dst_path)
                restored_count += 1
            except Exception as e:
                print(f"  ⚠️ {file_name} 이동 중 에러 발생: {e}")
        
        # 2. 파일을 다 꺼내서 빈 껍데기가 된 폴더 삭제
        try:
            os.rmdir(folder_path)
            removed_folders += 1
        except Exception as e:
            print(f"  ⚠️ [{item}] 폴더 삭제 실패 (안에 숨김 파일이 남아있을 수 있습니다): {e}")

# ==========================================
# 3. 결과 보고
# ==========================================
print(f"\n🎉 원상복구 완료!")
print(f"👉 총 {restored_count}장의 사진을 제자리로 돌려놓았고, {removed_folders}개의 그룹 폴더를 삭제했습니다.")
print("이제 군집 개수(NUM_CLUSTERS)를 변경하여 다시 분류를 시작하셔도 좋습니다!")

🚀 [C:\Users\user\Desktop\musinsa_images\musinsa_images\가디건] 복구(Rollback) 작업을 시작합니다...

📂 [Group_0] 폴더 해체 중...
📂 [Group_1] 폴더 해체 중...
📂 [Group_10] 폴더 해체 중...
📂 [Group_11] 폴더 해체 중...
📂 [Group_12] 폴더 해체 중...
📂 [Group_13] 폴더 해체 중...
📂 [Group_14] 폴더 해체 중...
📂 [Group_15] 폴더 해체 중...
📂 [Group_16] 폴더 해체 중...
📂 [Group_17] 폴더 해체 중...
📂 [Group_18] 폴더 해체 중...
📂 [Group_19] 폴더 해체 중...
📂 [Group_2] 폴더 해체 중...
📂 [Group_20] 폴더 해체 중...
📂 [Group_21] 폴더 해체 중...
📂 [Group_22] 폴더 해체 중...
📂 [Group_23] 폴더 해체 중...
📂 [Group_24] 폴더 해체 중...
📂 [Group_3] 폴더 해체 중...
📂 [Group_4] 폴더 해체 중...
📂 [Group_5] 폴더 해체 중...
📂 [Group_6] 폴더 해체 중...
📂 [Group_7] 폴더 해체 중...
📂 [Group_8] 폴더 해체 중...
📂 [Group_9] 폴더 해체 중...

🎉 원상복구 완료!
👉 총 3012장의 사진을 제자리로 돌려놓았고, 25개의 그룹 폴더를 삭제했습니다.
이제 군집 개수(NUM_CLUSTERS)를 변경하여 다시 분류를 시작하셔도 좋습니다!


In [1]:
import os
from PIL import Image
from ultralytics import YOLOWorld

# ==========================================
# 1. 경로 설정
# ==========================================
INPUT_BASE_DIR = r"C:\Users\user\Desktop\졸작 데이터셋"
OUTPUT_BASE_DIR = r"C:\Users\user\Desktop\졸작_A모델_학습용_크롭데이터"

# ==========================================
# 2. ⭐️ 핵심: 폴더별 전용 찾기 단어(Prompt) 매핑
# ==========================================
# 각 폴더를 검사할 때, YOLO가 오직 이 단어들에만 반응하도록 '시야를 좁혀주는' 역할입니다.
TARGET_MAP = {
    "1_shirt": ["shirt", "blouse", "button-down shirt"],
    "2_knit": ["sweater", "knitwear", "pullover"],
    "3_jacket": ["jacket", "blazer"],
    "4_outer": ["outerwear", "coat", "jacket"],
    "5_coat": ["coat", "trench coat", "overcoat"],
    "6_padding": ["puffer jacket", "down jacket", "padded jacket", "padding"],
    "7_zipup": ["zip-up hoodie", "track jacket", "windbreaker"],
    "8_sweatshirt": ["sweatshirt", "crewneck"],
    "9_hoodie": ["hoodie", "hooded sweatshirt"],
    "10_shortsleeve": ["t-shirt", "short sleeve shirt"]
}

print("🧠 YOLO-World AI 모델을 불러오는 중입니다...")
model = YOLOWorld('yolov8s-world.pt') 

model.to("cpu")

# ==========================================
# 3. 오토 크롭 실행
# ==========================================
def process_auto_crop():
    for folder_name in os.listdir(INPUT_BASE_DIR):
        input_folder_path = os.path.join(INPUT_BASE_DIR, folder_name)
        if not os.path.isdir(input_folder_path): continue
            
        output_folder_path = os.path.join(OUTPUT_BASE_DIR, folder_name)
        os.makedirs(output_folder_path, exist_ok=True)
        
        # 💡 핵심 로직: 현재 폴더 이름에 맞는 타겟 단어만 모델에 장착!
        # 만약 매핑 테이블에 없는 폴더면 기본값으로 "top clothing"을 찾게 함.
        target_prompts = TARGET_MAP.get(folder_name, ["top clothing"])
        model.set_classes(target_prompts)
        
        print(f"\n✂️ [{folder_name}] 폴더 크롭 시작! (탐지 타겟: {target_prompts})")
        
        files = [f for f in os.listdir(input_folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        success_count = 0
        
        for file_name in files:
            img_path = os.path.join(input_folder_path, file_name)
            save_path = os.path.join(output_folder_path, file_name)
            if os.path.exists(save_path):
                # print(f"  ⏩ {file_name} : 이미 크롭된 파일이 있어서 건너뜁니다.") # (너무 많이 뜨면 지저분하니 주석 처리)
                continue
            
           
            
            try:
                # conf=0.15로 설정하여 타겟 의류를 관대하게 찾음
                results = model.predict(img_path, conf=0.15, verbose=False)
                boxes = results[0].boxes
                
                if len(boxes) == 0:
                    print(f"  ⚠️ {file_name} : 해당 타겟 옷을 찾지 못해 스킵합니다.")
                    continue
                
                # 타겟 의류(예: 셔츠)만 찾은 상태에서 가장 큰 면적을 선택
                largest_box = None
                max_area = 0
                
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    area = (x2 - x1) * (y2 - y1)
                    
                    if area > max_area:
                        max_area = area
                        largest_box = [x1, y1, x2, y2]
                
                if largest_box:
                    x1, y1, x2, y2 = largest_box
                    padding = 15 
                    
                    img = Image.open(img_path).convert("RGB")
                    width, height = img.size
                    
                    crop_x1 = max(0, int(x1) - padding)
                    crop_y1 = max(0, int(y1) - padding)
                    crop_x2 = min(width, int(x2) + padding)
                    crop_y2 = min(height, int(y2) + padding)
                    
                    cropped_img = img.crop((crop_x1, crop_y1, crop_x2, crop_y2))
                    cropped_img.save(save_path)
                    success_count += 1
                    
            except Exception as e:
                print(f"  ❌ {file_name} 처리 중 에러: {e}")
                
        print(f"✅ [{folder_name}] 완료! (총 {len(files)}장 중 {success_count}장 크롭 성공)")

if __name__ == "__main__":
    print("🚀 [타겟 맞춤형 오토 크롭 파이프라인] 가동!")
    process_auto_crop()
    print("🎉 모든 작업이 완료되었습니다!")

🧠 YOLO-World AI 모델을 불러오는 중입니다...
🚀 [타겟 맞춤형 오토 크롭 파이프라인] 가동!

✂️ [10_outer_jacket] 폴더 크롭 시작! (탐지 타겟: ['top clothing'])
  ⚠️ 29CM_outer_100.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_101.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_102.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_103.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_104.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_107.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_108.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_109.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_110.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_111.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_112.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_113.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_115.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_116.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_117.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_118.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_120.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_122.jpg : 해당 타겟 옷을 찾지 못해 스킵합니다.
  ⚠️ 29CM_outer_12

In [1]:
import os
from PIL import Image
from ultralytics import YOLOWorld

# ==========================================
# 1. 경로 설정
# ==========================================
INPUT_BASE_DIR = r"C:\Users\user\Desktop\musinsa_images\musinsa_images"
OUTPUT_BASE_DIR = r"C:\Users\user\Desktop\졸작_A모델_추가크롭데이터"

# ==========================================
# 2. ⭐️ 긴급 수정: 프롬프트 범위 대폭 확장!
# ==========================================
TARGET_MAP = {
    # 💡 반바지 구출을 위해 '하프팬츠', '바지' 등의 포괄적 단어 대거 추가
    "반바지": ["shorts", "short pants", "half pants", "bermuda shorts", "pants"],
    
    # 💡 다른 바지들도 덜 잡힐까봐 포괄적 단어 추가
    "슬랙스": ["slacks", "dress pants", "trousers", "pants"],
    "청바지": ["jeans", "denim pants", "pants"],
    "코튼팬츠": ["chinos", "cotton pants", "pants", "trousers"],
    "트레이닝팬츠": ["sweatpants", "track pants", "joggers", "pants"],
    
    # 💡 기존 프롬프트 유지
    "가디건": ["cardigan"],
    "모자": ["hat", "cap", "beanie"],
    "무스탕": ["shearling jacket", "leather jacket", "mustang jacket"],
    "블레이저": ["blazer", "suit jacket"],
    "청자켓": ["denim jacket", "jean jacket"],
    "플리스": ["fleece jacket", "fleece"],
    "후드집업": ["zip-up hoodie", "hooded jacket"]
}

print("🧠 YOLO-World AI 모델을 불러오는 중입니다...")
model = YOLOWorld('yolov8s-world.pt') 
model.to("cpu") 

# ==========================================
# 3. 오토 크롭 실행
# ==========================================
def process_auto_crop():
    for folder_name in os.listdir(INPUT_BASE_DIR):
        input_folder_path = os.path.join(INPUT_BASE_DIR, folder_name)
        if not os.path.isdir(input_folder_path): continue
            
        output_folder_path = os.path.join(OUTPUT_BASE_DIR, folder_name)
        os.makedirs(output_folder_path, exist_ok=True)
        
        target_prompts = TARGET_MAP.get(folder_name, ["clothing"])
        model.set_classes(target_prompts)
        
        print(f"\n✂️ [{folder_name}] 구출 작전 시작! (탐지 타겟: {target_prompts})")
        
        files = [f for f in os.listdir(input_folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        success_count = 0
        skip_count = 0
        
        for file_name in files:
            img_path = os.path.join(input_folder_path, file_name)
            save_path = os.path.join(output_folder_path, file_name)
            
            # 이어하기 스킵 (이미 잘린 5장은 무시하고 안 잘린 것만 검사)
            if os.path.exists(save_path):
                skip_count += 1
                continue
            
            try:
                # 🚨 긴급 수정: 허들을 대폭 낮춤 (conf=0.15 -> 0.05)
                # "아주 쪼끔만 반바지 같아도 일단 네모 박스를 쳐라!"
                results = model.predict(img_path, conf=0.05, verbose=False)
                boxes = results[0].boxes
                
                if len(boxes) == 0:
                    continue
                
                largest_box = None
                max_area = 0
                
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    area = (x2 - x1) * (y2 - y1)
                    if area > max_area:
                        max_area = area
                        largest_box = [x1, y1, x2, y2]
                
                if largest_box:
                    x1, y1, x2, y2 = largest_box
                    padding = 15 
                    
                    img = Image.open(img_path).convert("RGB")
                    width, height = img.size
                    
                    crop_x1 = max(0, int(x1) - padding)
                    crop_y1 = max(0, int(y1) - padding)
                    crop_x2 = min(width, int(x2) + padding)
                    crop_y2 = min(height, int(y2) + padding)
                    
                    cropped_img = img.crop((crop_x1, crop_y1, crop_x2, crop_y2))
                    cropped_img.save(save_path)
                    success_count += 1
                    
            except Exception as e:
                print(f"  ❌ {file_name} 처리 중 에러: {e}")
                
        print(f"✅ [{folder_name}] 완료! (새로 구출됨: {success_count}장 | 기존 성공분 스킵: {skip_count}장)")

if __name__ == "__main__":
    print("🚀 [데이터 구출 작전] 가동!")
    process_auto_crop()
    print("🎉 구출 작업이 완료되었습니다!")

🧠 YOLO-World AI 모델을 불러오는 중입니다...
🚀 [데이터 구출 작전] 가동!

✂️ [가디건] 구출 작전 시작! (탐지 타겟: ['cardigan'])
✅ [가디건] 완료! (새로 구출됨: 1392장 | 기존 성공분 스킵: 325장)

✂️ [모자] 구출 작전 시작! (탐지 타겟: ['hat', 'cap', 'beanie'])
✅ [모자] 완료! (새로 구출됨: 246장 | 기존 성공분 스킵: 2652장)

✂️ [무스탕] 구출 작전 시작! (탐지 타겟: ['shearling jacket', 'leather jacket', 'mustang jacket'])
✅ [무스탕] 완료! (새로 구출됨: 376장 | 기존 성공분 스킵: 1036장)

✂️ [반바지] 구출 작전 시작! (탐지 타겟: ['shorts', 'short pants', 'half pants', 'bermuda shorts', 'pants'])
✅ [반바지] 완료! (새로 구출됨: 134장 | 기존 성공분 스킵: 5장)

✂️ [블레이저] 구출 작전 시작! (탐지 타겟: ['blazer', 'suit jacket'])
✅ [블레이저] 완료! (새로 구출됨: 1862장 | 기존 성공분 스킵: 653장)

✂️ [슬랙스] 구출 작전 시작! (탐지 타겟: ['slacks', 'dress pants', 'trousers', 'pants'])
✅ [슬랙스] 완료! (새로 구출됨: 1131장 | 기존 성공분 스킵: 0장)

✂️ [청바지] 구출 작전 시작! (탐지 타겟: ['jeans', 'denim pants', 'pants'])
✅ [청바지] 완료! (새로 구출됨: 341장 | 기존 성공분 스킵: 0장)

✂️ [청자켓] 구출 작전 시작! (탐지 타겟: ['denim jacket', 'jean jacket'])
✅ [청자켓] 완료! (새로 구출됨: 1731장 | 기존 성공분 스킵: 0장)

✂️ [코튼팬츠] 구출 작전 시작! (탐지 타겟: ['chinos', 'cotton pants', 'p

In [ ]:
import os
from PIL import Image
from ultralytics import YOLOWorld

# ==========================================
# 1. 경로 설정
# ==========================================
# 🚨 새로 알려주신 입력 폴더 경로
INPUT_BASE_DIR = r"C:\Users\user\Desktop\musinsa_images\musinsa_images"

# 🚨 크롭된 이미지를 저장할 새로운 폴더
OUTPUT_BASE_DIR = r"C:\Users\user\Desktop\졸작_A모델_추가크롭데이터"

# ==========================================
# 2. ⭐️ 핵심: 새 카테고리별 전용 찾기 단어(Prompt) 매핑
# ==========================================
# 한글 폴더명에 맞춰 YOLO가 가장 잘 인식하는 영어 단어들로 세팅했습니다.
TARGET_MAP = {
    "가디건": ["cardigan"],
    "모자": ["hat", "cap", "beanie"],
    "무스탕": ["shearling jacket", "leather jacket", "mustang jacket"],
    "반바지": ["shorts", "short pants"],
    "블레이저": ["blazer", "suit jacket"],
    "슬랙스": ["slacks", "dress pants", "trousers"],
    "청바지": ["jeans", "denim pants"],
    "청자켓": ["denim jacket", "jean jacket"],
    "코튼팬츠": ["chinos", "cotton pants", "pants"],
    "트레이닝팬츠": ["sweatpants", "track pants", "joggers"],
    "플리스": ["fleece jacket", "fleece"],
    "후드집업": ["zip-up hoodie", "hooded jacket"]
}

print("🧠 YOLO-World AI 모델을 불러오는 중입니다...")
model = YOLOWorld('yolov8s-world.pt') 

# 🚨 CPU 강제 할당 (GPU 충돌 에러 방지)
model.to("cpu") 

# ==========================================
# 3. 오토 크롭 실행
# ==========================================
def process_auto_crop():
    for folder_name in os.listdir(INPUT_BASE_DIR):
        input_folder_path = os.path.join(INPUT_BASE_DIR, folder_name)
        if not os.path.isdir(input_folder_path): continue
            
        output_folder_path = os.path.join(OUTPUT_BASE_DIR, folder_name)
        os.makedirs(output_folder_path, exist_ok=True)
        
        # 💡 매핑 테이블에 없는 폴더면 기본값으로 "clothing"을 찾게 함.
        target_prompts = TARGET_MAP.get(folder_name, ["clothing"])
        model.set_classes(target_prompts)
        
        print(f"\n✂️ [{folder_name}] 폴더 크롭 시작! (탐지 타겟: {target_prompts})")
        
        files = [f for f in os.listdir(input_folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        success_count = 0
        skip_count = 0
        
        for file_name in files:
            img_path = os.path.join(input_folder_path, file_name)
            save_path = os.path.join(output_folder_path, file_name)
            
            # 🚨 멈췄다 다시 켤 때 시간을 아껴주는 이어하기(Skip) 로직!
            if os.path.exists(save_path):
                skip_count += 1
                continue
            
            try:
                # conf=0.15로 설정하여 타겟 의류를 관대하게 찾음
                results = model.predict(img_path, conf=0.15, verbose=False)
                boxes = results[0].boxes
                
                if len(boxes) == 0:
                    # 너무 많이 출력되면 지저분하므로 프린트는 생략해도 좋습니다
                    # print(f"  ⚠️ {file_name} : 타겟 옷을 찾지 못함")
                    continue
                
                # 타겟 의류만 찾은 상태에서 가장 큰 면적을 선택
                largest_box = None
                max_area = 0
                
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    area = (x2 - x1) * (y2 - y1)
                    
                    if area > max_area:
                        max_area = area
                        largest_box = [x1, y1, x2, y2]
                
                if largest_box:
                    x1, y1, x2, y2 = largest_box
                    padding = 15 # 안전 여백 15픽셀
                    
                    img = Image.open(img_path).convert("RGB")
                    width, height = img.size
                    
                    crop_x1 = max(0, int(x1) - padding)
                    crop_y1 = max(0, int(y1) - padding)
                    crop_x2 = min(width, int(x2) + padding)
                    crop_y2 = min(height, int(y2) + padding)
                    
                    cropped_img = img.crop((crop_x1, crop_y1, crop_x2, crop_y2))
                    cropped_img.save(save_path)
                    success_count += 1
                    
            except Exception as e:
                print(f"  ❌ {file_name} 처리 중 에러: {e}")
                
        print(f"✅ [{folder_name}] 완료! (성공: {success_count}장 | 이어하기 스킵: {skip_count}장)")

if __name__ == "__main__":
    print("🚀 [추가 카테고리 맞춤형 오토 크롭 파이프라인] 가동!")
    process_auto_crop()
    print("🎉 모든 작업이 완료되었습니다!")

🧠 YOLO-World AI 모델을 불러오는 중입니다...
🚀 [추가 카테고리 맞춤형 오토 크롭 파이프라인] 가동!

✂️ [가디건] 폴더 크롭 시작! (탐지 타겟: ['cardigan'])
✅ [가디건] 완료! (성공: 325장 | 이어하기 스킵: 0장)

✂️ [모자] 폴더 크롭 시작! (탐지 타겟: ['hat', 'cap', 'beanie'])
✅ [모자] 완료! (성공: 2652장 | 이어하기 스킵: 0장)

✂️ [무스탕] 폴더 크롭 시작! (탐지 타겟: ['shearling jacket', 'leather jacket', 'mustang jacket'])
✅ [무스탕] 완료! (성공: 1036장 | 이어하기 스킵: 0장)

✂️ [반바지] 폴더 크롭 시작! (탐지 타겟: ['shorts', 'short pants'])
✅ [반바지] 완료! (성공: 5장 | 이어하기 스킵: 0장)

✂️ [블레이저] 폴더 크롭 시작! (탐지 타겟: ['blazer', 'suit jacket'])


KeyboardInterrupt: 